In [1]:
import cv2
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO
import easyocr
from yt_dlp import YoutubeDL
import imageio_ffmpeg

In [2]:
REPO_ROOT = Path(os.getcwd()).parent

YOUTUBE_LINK       = "https://www.youtube.com/watch?v=3vBHgZI1Yv8"
VIDEOS_PATH        = "videos"
CROP_MODEL_PATH = "/work/classtmp/ryanbog/robots/FIRST-Robotics-Competition-Data-Challenge/models/crop_scoreboard.pt"
INFO_MODEL_PATH = "/work/classtmp/ryanbog/robots/FIRST-Robotics-Competition-Data-Challenge/models/extract_scoreboard_info.pt" 
SCOREBOARD_SKIP    = 15          # process every Nth frame for scoreboard
DELETE_VIDEO       = False

VIDEO_PATH         = ("/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos"
                      "/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4")
ROBOT_MODEL_PATH   = REPO_ROOT / "yolov8_model"    / "best_tuned_yolov8.pt"
NUMBER_MODEL_PATH  = REPO_ROOT / "robot_numbers"   / "runs" / "detect" / "train" / "weights" / "best.pt"
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers"       / "botsort_custom.yaml"

REEF_CLASS_ID        = 0
ROBOT_CLASS_ID       = 1


BLUE_NUMBER_CLASS_ID = 0
RED_NUMBER_CLASS_ID  = 1

BLUE_IDS = [3928, 9570, 5809]
RED_IDS  = [8825, 1736, 2357]

TARGET_POINTS = 5

SEARCH_WINDOW = 10

TRACKING_STRIDE = 3
OCR_STRIDE      = 15

DEVICE = 0   

os.makedirs(VIDEOS_PATH, exist_ok=True)

In [ ]:
# ──────────────────────────────────────────────
# CELL 3 — Scoreboard helpers
# ──────────────────────────────────────────────
crop_model = YOLO(CROP_MODEL_PATH)
info_model = YOLO(INFO_MODEL_PATH)
reader     = easyocr.Reader(['en'], gpu=(DEVICE != "cpu"))


def preprocess_for_ocr(roi):
    """Resize, boost contrast, and threshold a BGR ROI for OCR."""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    gray = cv2.convertScaleAbs(gray, alpha=1.8, beta=10)
    _, thresh     = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh_inv    = cv2.bitwise_not(thresh)
    return [thresh, thresh_inv]


def read_best_text(images, allowlist):
    best_text, best_conf = None, 0
    for img in images:
        for (_, text, conf) in reader.readtext(img, allowlist=allowlist,
                                               detail=1, paragraph=False):
            if conf > best_conf:
                best_conf, best_text = conf, text
    return best_text


def read_number(img):
    text = read_best_text(preprocess_for_ocr(img), '0123456789')
    if text is None:
        return None
    text = re.sub(r"\D", "", text)
    return int(text) if text else None


def read_timer(img):
    text = read_best_text(preprocess_for_ocr(img), '0123456789:')
    if text is None:
        return None
    m = re.search(r"\d{1,2}:\d{2}", text)
    return m.group(0) if m else None


def download_video(url, output_dir):
    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
        "ffmpeg_location": imageio_ffmpeg.get_ffmpeg_exe(),
        "merge_output_format": "mp4",
    }
    with YoutubeDL(ydl_opts) as ydl:
        info     = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        return os.path.splitext(os.path.basename(filename))[0] + ".mp4"

In [ ]:
# ──────────────────────────────────────────────
# CELL 4 — Scoreboard extraction → output_df
#
# Fixes vs original:
#  • Null-safe score/timer handling: frames where only ONE
#    of (blue_score, red_score) is None are carried forward
#    using the last known value instead of being dropped.
#  • pending_row logic kept; timer-missing frames are held
#    and resolved when the timer reappears.
#  • Alliance-side midpoint guard: if blue/red centre_x is
#    None we still fall back gracefully instead of crashing.
# ──────────────────────────────────────────────
video_filename = download_video(YOUTUBE_LINK, VIDEOS_PATH)
video_path     = os.path.join(VIDEOS_PATH, video_filename)

rows       = []
prev_blue  = None
prev_red   = None
pending_row = None
is_auto    = True
youtube_url = YOUTUBE_LINK

cap       = cv2.VideoCapture(video_path)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % SCOREBOARD_SKIP != 0:
        frame_idx += 1
        continue

    crop_results = crop_model(frame, device=DEVICE)[0]
    if len(crop_results.boxes) == 0:
        frame_idx += 1
        continue

    x1, y1, x2, y2 = map(int, crop_results.boxes.xyxy[0])
    scoreboard      = frame[y1:y2, x1:x2]

    info_results = info_model(scoreboard, device=DEVICE)[0]

    blue_score   = None
    red_score    = None
    timer        = None
    blue_center_x = None
    red_center_x  = None
    team_data    = []

    for b in info_results.boxes:
        cls_id = int(b.cls[0])
        label  = info_model.names[cls_id]
        bx1, by1, bx2, by2 = map(int, b.xyxy[0])
        region   = scoreboard[by1:by2, bx1:bx2]
        x_center = (bx1 + bx2) / 2

        if label == "blue_score":
            blue_score    = read_number(region)
            blue_center_x = x_center
        elif label == "red_score":
            red_score     = read_number(region)
            red_center_x  = x_center
        elif label == "timer":
            timer = read_timer(region)
        elif label == "team_number":
            num = read_number(region)
            if num is not None:
                team_data.append((num, x_center))

    # ── Null-safe score fill: carry forward last known values ──
    # This prevents frames where OCR misses one score from being
    # treated as a score change or dropped entirely.
    if blue_score is None and prev_blue is not None:
        blue_score = prev_blue
    if red_score is None and prev_red is not None:
        red_score = prev_red

    # Drop frames where we still have no score information at all
    if blue_score is None and red_score is None:
        frame_idx += 1
        continue

    # Monotone guard: scores must not decrease
    if prev_blue is not None and blue_score is not None and blue_score < prev_blue:
        frame_idx += 1
        continue
    if prev_red is not None and red_score is not None and red_score < prev_red:
        frame_idx += 1
        continue

    # Skip unchanged frames
    if prev_blue == blue_score and prev_red == red_score:
        frame_idx += 1
        continue

    # Detect transition from auto → teleop
    if timer is not None and is_auto:
        try:
            if int(timer.split(":")[0]) == 2:
                is_auto = False
        except Exception:
            pass

    # ── Team number → alliance assignment ──
    red_location = None
    if blue_center_x is not None and red_center_x is not None:
        red_location = "right" if red_center_x > blue_center_x else "left"

    blue_teams = []
    red_teams  = []

    if blue_center_x is not None and red_center_x is not None:
        midpoint = (blue_center_x + red_center_x) / 2
        for num, x in team_data:
            if red_location == "right":
                if x < midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))
            else:
                if x > midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))
    else:
        # Fallback: no positional info → skip team assignment this frame
        # (team numbers will remain empty lists; scoring lookup still works)
        pass

    blue_team_numbers = [n for n, _ in sorted(blue_teams, key=lambda t: t[1])[:3]]
    red_team_numbers  = [n for n, _ in sorted(red_teams,  key=lambda t: t[1])[:3]]

    current_row = {
        "Frame":            frame_idx,
        "blue_score":       blue_score,
        "red_score":        red_score,
        "timer":            timer,
        "is_auto":          is_auto,
        "red_location":     red_location,
        "blue_team_numbers": blue_team_numbers,
        "red_team_numbers":  red_team_numbers,
        "youtube_link":     youtube_url,
    }

    # ── Timer-missing buffering ──
    if timer is None:
        if pending_row is None:
            pending_row = current_row
        frame_idx += 1
        continue

    # Timer present — resolve any buffered row
    if (
        pending_row is not None
        and blue_score == pending_row["blue_score"]
        and red_score  == pending_row["red_score"]
    ):
        # Same scores with timer now visible: upgrade pending with timer
        rows.append(current_row)
        pending_row = None
    else:
        # Scores changed — flush pending first
        if pending_row is not None:
            rows.append(pending_row)
            pending_row = None
        rows.append(current_row)

    prev_blue = blue_score
    prev_red  = red_score
    frame_idx += 1

if pending_row is not None:
    rows.append(pending_row)

cap.release()

if DELETE_VIDEO and os.path.exists(video_path):
    os.remove(video_path)

output_df = pd.DataFrame(rows)
print(f"Scoreboard rows captured: {len(output_df)}")
output_df.head()

[youtube] Extracting URL: https://www.youtube.com/watch?v=NSWVoO4ZDEs
[youtube] NSWVoO4ZDEs: Downloading webpage


[youtube] NSWVoO4ZDEs: Downloading android vr player API JSON
[info] NSWVoO4ZDEs: Downloading 1 format(s): 137+251
[download] videos/Final 1 - 2025 Rocket City Regional.mp4 has already been downloaded

0: 384x640 (no detections), 42.8ms
Speed: 5.6ms preprocess, 42.8ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.3ms
Speed: 1.6ms preprocess, 6.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.6ms
Speed: 1.4ms preprocess, 6.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.6ms
Speed: 1.3ms preprocess, 6.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.4ms
Speed: 1.2ms preprocess, 6.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.3ms
Speed: 1.2ms preprocess, 6.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640

,Frame,blue_score,red_score,timer,is_auto,red_location,blue_team_numbers,red_team_numbers,youtube_link
0,105,0,0,0:00,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
1,315,9,0,0:13,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
2,345,9,7,0:12,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
3,360,16,14,0:11,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
4,375,23,17,0:11,True,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs


In [ ]:
# ──────────────────────────────────────────────
# CELL 5 — Load robot / number models
# ──────────────────────────────────────────────
robot_model  = YOLO(ROBOT_MODEL_PATH)
number_model = YOLO(NUMBER_MODEL_PATH)

In [10]:
# ──────────────────────────────────────────────
# CELL 6 — BoT-SORT tracking
# Returns:
#   tracking_results : list of {frame, track_id, box, crop}
#   frame_data       : dict frame → {robots, reef}
#   reef_results     : list of {frame, box}
# ──────────────────────────────────────────────
def run_botsort(video_path, tracker_path, stride=1):
    print("Running BoT-SORT tracking...")

    results_gen = robot_model.track(
        source=video_path,
        tracker=tracker_path,
        stream=True,
        persist=True,
        conf=0.35,
        device=DEVICE,
        vid_stride=stride,
        verbose=False,
    )

    tracking_results = []
    reef_results     = []

    for frame_i, result in enumerate(results_gen):
        if result.boxes is None or result.boxes.xyxy is None:
            continue

        ids = result.boxes.id
        if ids is None:
            continue

        boxes   = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        ids     = ids.cpu().numpy().astype(int)
        frame   = result.plot()

        actual_frame = frame_i * stride

        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes[i]
            cls      = classes[i]
            track_id = int(ids[i])

            if cls == ROBOT_CLASS_ID:
                crop = frame[int(y1):int(y2), int(x1):int(x2)].copy()
                tracking_results.append({
                    "frame":    actual_frame,
                    "track_id": track_id,
                    "box":      [float(x1), float(y1), float(x2), float(y2)],
                    "crop":     crop,
                })

            elif cls == REEF_CLASS_ID:
                reef_results.append({
                    "frame": actual_frame,
                    "box":   [float(x1), float(y1), float(x2), float(y2)],
                })

    print(f"Tracking complete: {len(tracking_results)} robot detections, "
          f"{len(reef_results)} reef detections.")
    return tracking_results, reef_results

In [11]:
# ──────────────────────────────────────────────
# CELL 7 — Image preprocessing helpers for OCR
# ──────────────────────────────────────────────
def estimate_angle(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0
    cnt   = max(contours, key=cv2.contourArea)
    angle = cv2.minAreaRect(cnt)[-1]
    if angle < -45:
        angle += 90
    return angle


def rotate_image(img, angle):
    h, w   = img.shape[:2]
    M      = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess_variants(crop):
    gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    variants = []
    for t in [150, 165, 180, 195, 210]:
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        variants.append(th)
    variants.append(cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2))
    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(otsu)
    eq = cv2.equalizeHist(gray)
    _, th_eq = cv2.threshold(eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(th_eq)
    return variants

In [12]:
# ──────────────────────────────────────────────
# CELL 8 — OCR for team numbers on robot crops
# ──────────────────────────────────────────────
def read_number_from_image(img):
    variants = preprocess_variants(img)
    angle    = estimate_angle(img)
    guesses  = []

    for processed in variants:
        rotated = rotate_image(processed, angle)
        results = reader.readtext(rotated, allowlist="0123456789",
                                  detail=1, paragraph=False)
        results = [r for r in results if r[2] > 0.5]
        if results:
            guesses.append(results[0][1])

    if not guesses:
        return None

    c         = Counter(guesses)
    max_count = max(c.values())
    tied      = [num for num, count in c.items() if count == max_count]
    return max(tied, key=lambda x: len(str(x)))

In [13]:
# ──────────────────────────────────────────────
# CELL 9 — Fuzzy number-matching helpers
# ──────────────────────────────────────────────
def levenshtein(a, b):
    a, b = str(a), str(b)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1): dp[i][0] = i
    for j in range(len(b) + 1): dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost    = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[-1][-1]


def similarity(a, b):
    if not a or not b:
        return 0
    return 1 - levenshtein(a, b) / max(len(str(a)), len(str(b)))


def alignment_score(a, b):
    a, b = str(a), str(b)
    best = 0
    for shift in range(-len(b), len(a) + 1):
        matches = sum(
            1 for i in range(len(a))
            if 0 <= i - shift < len(b) and a[i] == b[i - shift]
        )
        best = max(best, matches)
    return best / max(len(a), len(b))


def combined_score(a, b, w1=0.7, w2=0.3):
    return w1 * similarity(a, b) + w2 * alignment_score(a, b)


def match_number_single(detected, nums):
    if detected is None:
        return None
    best_score, match = 0.5, None
    for n in nums:
        s = combined_score(detected, n)
        if s > best_score:
            best_score, match = s, n
    return match

In [14]:
# ──────────────────────────────────────────────
# CELL 10 — Read team numbers from robot crops
# ──────────────────────────────────────────────
def read_numbers(tracking_results, blue_ids=(), red_ids=(), stride=1):
    """OCR every robot crop and attempt to match to a known team number."""
    results       = []
    current_frame = None
    frame_data    = []

    for entry in tracking_results:
        frame_idx  = entry["frame"]
        robot_crop = entry["crop"]

        if frame_idx % stride != 0:
            continue

        if current_frame is None:
            current_frame = frame_idx

        if frame_idx != current_frame:
            results.append({"frame": current_frame, "detections": frame_data})
            frame_data    = []
            current_frame = frame_idx

        if robot_crop is None or robot_crop.size == 0:
            continue

        number_results = number_model(robot_crop, verbose=False)[0]

        for nbox in number_results.boxes:
            cls_id = int(nbox.cls[0])
            if   cls_id == BLUE_NUMBER_CLASS_ID: team_list = list(blue_ids)
            elif cls_id == RED_NUMBER_CLASS_ID:  team_list = list(red_ids)
            else: continue

            if nbox.conf[0] < 0.4:
                continue

            nx1, ny1, nx2, ny2 = map(int, nbox.xyxy[0])
            pad = 5
            h2, w2 = robot_crop.shape[:2]
            nx1, ny1 = max(0, nx1 - pad), max(0, ny1 - pad)
            nx2, ny2 = min(w2, nx2 + pad), min(h2, ny2 + pad)

            if (nx2 - nx1) < 30 or (ny2 - ny1) < 15:
                continue

            number_crop = robot_crop[ny1:ny2, nx1:nx2]
            if number_crop.size == 0:
                continue

            detected = read_number_from_image(number_crop)
            matched  = match_number_single(detected, team_list)

            frame_data.append({
                "detected":      detected,
                "matched":       matched,
                "track_id":      entry["track_id"],
                "box":           entry["box"],
                "alliance_hint": "blue" if cls_id == BLUE_NUMBER_CLASS_ID else "red",
            })

    if frame_data:
        results.append({"frame": current_frame, "detections": frame_data})

    return results

In [15]:
# ──────────────────────────────────────────────
# CELL 11 — Resolve track_id → team number + alliance
#
# Null-safe improvements vs original:
#  • Frames with no OCR result are skipped, not crashed on.
#  • Alliance assignment falls back to alliance_hint when
#    OCR vote is tied (was previously left as None).
#  • Elimination pass fills any robot whose alliance is known
#    but team number is still unresolved at the end.
# ──────────────────────────────────────────────
def resolve_labels(ocr_output, blue_ids, red_ids):
    blue_ids = list(blue_ids)
    red_ids  = list(red_ids)
    all_ids  = blue_ids + red_ids

    # --- Step 1: vote for alliance per track ---
    track_alliance_votes = defaultdict(lambda: {"blue": 0, "red": 0})

    for frame in ocr_output:
        for det in frame["detections"]:
            tid   = det["track_id"]
            match = det.get("matched")
            hint  = det.get("alliance_hint")

            if hint == "blue":  track_alliance_votes[tid]["blue"] += 2
            elif hint == "red": track_alliance_votes[tid]["red"]  += 2

            if match in blue_ids: track_alliance_votes[tid]["blue"] += 1
            elif match in red_ids: track_alliance_votes[tid]["red"]  += 1

    track_alliance = {}
    for tid, votes in track_alliance_votes.items():
        if votes["blue"] > votes["red"]:
            track_alliance[tid] = "blue"
        elif votes["red"] > votes["blue"]:
            track_alliance[tid] = "red"
        else:
            # Tie: fall back to hint majority (covers null-OCR frames)
            track_alliance[tid] = None

    # --- Step 2: weighted vote for team number per track ---
    track_memory = defaultdict(list)

    for frame in ocr_output:
        for det in frame["detections"]:
            tid      = det["track_id"]
            match    = det.get("matched")
            alliance = track_alliance.get(tid)

            if match is None:
                continue
            if alliance == "blue" and match in blue_ids:
                track_memory[tid].append(match)
            elif alliance == "red" and match in red_ids:
                track_memory[tid].append(match)

    final_labels = {}

    for tid, nums in track_memory.items():
        if not nums:
            continue
        scores = defaultdict(float)
        for i, num in enumerate(nums):
            scores[num] += 0.9 ** (len(nums) - i)
        best         = max(scores, key=scores.get)
        total_weight = sum(scores.values())
        if scores[best] / total_weight >= 0.5:
            final_labels[tid] = best

    # --- Step 3: elimination to fill remaining unknowns ---
    def eliminate(alliance_key, ids_list):
        assigned   = {v for k, v in final_labels.items()
                      if track_alliance.get(k) == alliance_key}
        remaining  = [i for i in ids_list if i not in assigned]
        unlabeled  = [t for t, a in track_alliance.items()
                      if a == alliance_key and t not in final_labels]
        if len(unlabeled) == len(remaining):
            for t, rid in zip(unlabeled, remaining):
                final_labels[t] = rid

    eliminate("blue", blue_ids)
    eliminate("red",  red_ids)

    # Sanity-check: remove cross-alliance mismatches
    for tid in list(final_labels):
        alliance = track_alliance.get(tid)
        label    = final_labels[tid]
        if alliance == "blue" and label not in blue_ids:
            del final_labels[tid]
        elif alliance == "red" and label not in red_ids:
            del final_labels[tid]

    print("Final track labels :", final_labels)
    print("Track alliances    :", {k: v for k, v in track_alliance.items() if v})
    return final_labels, track_alliance

In [16]:
# ──────────────────────────────────────────────
# CELL 12 — Build per-frame data structure
#           (robots + reef, keyed by frame index)
# ──────────────────────────────────────────────
def build_frame_data(tracking_results, reef_detections, final_labels, track_alliance):
    frame_data = defaultdict(lambda: {"robots": [], "reef": None})

    for entry in tracking_results:
        f   = entry["frame"]
        tid = entry["track_id"]
        frame_data[f]["robots"].append({
            "track_id": tid,
            "team":     final_labels.get(tid),       # may be None
            "alliance": track_alliance.get(tid),     # 'blue'|'red'|None
            "bbox":     entry["box"],
        })

    for r in reef_detections:
        frame_data[r["frame"]]["reef"] = r["box"]

    return frame_data

In [17]:
# ──────────────────────────────────────────────
# CELL 13 — Extract only +5 point scoring events
#           from the scoreboard DataFrame.
# ──────────────────────────────────────────────
def get_scoring_events(output_df, target_points=5):
    """
    Returns a list of dicts:
        {frame, alliance, points}
    Only events where the score increases by exactly `target_points`.
    """
    events    = []
    prev_blue = 0
    prev_red  = 0

    for _, row in output_df.iterrows():
        f         = int(row["Frame"])
        blue_now  = row["blue_score"] if pd.notna(row["blue_score"]) else prev_blue
        red_now   = row["red_score"]  if pd.notna(row["red_score"])  else prev_red

        blue_delta = int(blue_now) - int(prev_blue)
        red_delta  = int(red_now)  - int(prev_red)

        if blue_delta == target_points:
            events.append({"frame": f, "alliance": "blue", "points": blue_delta})
        if red_delta  == target_points:
            events.append({"frame": f, "alliance": "red",  "points": red_delta})

        prev_blue = int(blue_now)
        prev_red  = int(red_now)

    print(f"Found {len(events)} events of exactly +{target_points} points.")
    return events

In [18]:
# ──────────────────────────────────────────────
# CELL 14 — Compute points per robot
#
# Key logic: for a +5 event for alliance X, find the robot from
# alliance X that is closest to the TOP QUADRANT of the reef
# within [frame-SEARCH_WINDOW, frame].
#
# "Top quadrant" = upper half of the reef bounding box
#   reef_top_center = ((x1+x2)/2, y1 + (y2-y1)*0.25)
#   i.e. the centre of the top 25 % of the reef box.
# ──────────────────────────────────────────────
def reef_top_quadrant_center(reef_box):
    """Return the centroid of the top quarter of a reef bounding box."""
    x1, y1, x2, y2 = reef_box
    cx  = (x1 + x2) / 2
    cy  = y1 + (y2 - y1) * 0.25       # 25 % down from the top
    return np.array([cx, cy])


def robot_center(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])


def compute_points(frame_data, scoring_events, track_alliance,
                   search_window=SEARCH_WINDOW):
    """
    For each +5 scoring event, credit the robot of the correct alliance
    that is closest to the reef's top quadrant in the search window.
    """
    robot_points = defaultdict(int)

    for event in scoring_events:
        best_robot = None
        best_dist  = float("inf")
        event_alliance = event["alliance"]

        for f in range(event["frame"] - search_window, event["frame"] + 1):
            if f not in frame_data:
                continue

            reef = frame_data[f]["reef"]
            if reef is None:
                continue

            reef_tc = reef_top_quadrant_center(reef)

            for r in frame_data[f]["robots"]:
                robot_alliance = r.get("alliance")

                # ── Alliance filter ──────────────────────────────────────
                # Only credit a robot from the scoring alliance.
                # If alliance is unknown (None), skip — don't misattribute.
                if robot_alliance is None:
                    continue
                if robot_alliance != event_alliance:
                    continue
                # ─────────────────────────────────────────────────────────

                d = np.linalg.norm(robot_center(r["bbox"]) - reef_tc)
                if d < best_dist:
                    best_dist  = d
                    best_robot = r["track_id"]

        if best_robot is not None:
            robot_points[best_robot] += event["points"]

    return robot_points

In [19]:
# ──────────────────────────────────────────────
# CELL 15 — Classify offense / defense
# ──────────────────────────────────────────────
def classify_roles(frame_data, robot_points):
    robot_distances = defaultdict(list)

    for f, data in frame_data.items():
        if data["reef"] is None:
            continue
        reef_c = reef_top_quadrant_center(data["reef"])

        for r in data["robots"]:
            robot_distances[r["track_id"]].append(
                np.linalg.norm(robot_center(r["bbox"]) - reef_c)
            )

    roles = {}
    for rid in robot_points:
        avg_dist   = np.mean(robot_distances[rid]) if robot_distances[rid] else 999
        roles[rid] = "offense" if (robot_points[rid] >= 5 and avg_dist < 200) else "defense"

    return roles

In [20]:
# ──────────────────────────────────────────────
# CELL 16 — Build final 6-robot results table
#
# Output: exactly 6 rows (3 blue + 3 red) regardless of
# how many unique track IDs were seen.  Robots that were
# tracked but scored 0 are included with points_scored=0.
# ──────────────────────────────────────────────
def build_results(robot_points, roles, final_labels, track_alliance,
                  blue_ids, red_ids):
    """
    Returns a DataFrame with columns:
        team_number | alliance | points_scored | role
    Exactly 3 blue rows and 3 red rows.
    """
    # Build a reverse map: team_number → alliance (from scoreboard gold truth)
    num_to_alliance = {}
    for n in blue_ids: num_to_alliance[n] = "blue"
    for n in red_ids:  num_to_alliance[n] = "red"

    # Build a reverse map: team_number → track_id (first occurrence wins)
    num_to_track = {v: k for k, v in final_labels.items()}

    rows_out = []
    for num in blue_ids + red_ids:
        tid  = num_to_track.get(num)            # may be None if never labelled
        pts  = robot_points.get(tid, 0) if tid else 0
        role = roles.get(tid, "defense")        if tid else "defense"
        rows_out.append({
            "team_number":   num,
            "alliance":      num_to_alliance[num],
            "points_scored": pts,
            "role":          role,
        })

    results_df = pd.DataFrame(rows_out)
    results_df = results_df.sort_values(
        ["alliance", "points_scored"], ascending=[True, False]
    ).reset_index(drop=True)
    return results_df

In [21]:
# ──────────────────────────────────────────────
# CELL 17 — RUN FULL PIPELINE
# ──────────────────────────────────────────────
output_dir = REPO_ROOT / "tracking_output"
output_dir.mkdir(exist_ok=True)

# 1. BoT-SORT: detect & track robots + reef
tracking_results, reef_results = run_botsort(
    VIDEO_PATH, CUSTOM_TRACKER_PATH, stride=TRACKING_STRIDE
)

# 2. OCR: read team numbers from robot crops
print("Reading team numbers via OCR...")
ocr_output = read_numbers(
    tracking_results, BLUE_IDS, RED_IDS, stride=OCR_STRIDE
)

# 3. Resolve track_id → team number + alliance
print("Resolving track labels...")
final_labels, track_alliance = resolve_labels(ocr_output, BLUE_IDS, RED_IDS)

# 4. Build per-frame data (robots + reef)
frame_data = build_frame_data(
    tracking_results, reef_results, final_labels, track_alliance
)

# 5. Extract only +5 point events from scoreboard CSV
events = get_scoring_events(output_df, target_points=TARGET_POINTS)

# 6. Credit points — robot closest to reef TOP quadrant, correct alliance
robot_points = compute_points(frame_data, events, track_alliance)

# 7. Classify offense / defense
roles = classify_roles(frame_data, robot_points)

# 8. Assemble final 6-robot table
results_df = build_results(
    robot_points, roles, final_labels, track_alliance, BLUE_IDS, RED_IDS
)

print("\n══════════════════════════════════════")
print("  SCORING SUMMARY  (+5 pt reef events)")
print("══════════════════════════════════════")
print(results_df.to_string(index=False))
results_df

Running BoT-SORT tracking...
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
Tracking complete: 7202 robot detections, 3264 reef detections.
Reading team numbers via OCR...
Resolving track labels...
Final track labels : {}
Track alliances    : {}
Found 11 events of exactly +5 points.

══════════════════════════════════════
  SCORING SUMMARY  (+5 pt reef events)
══════════════════════════════════════
 team_number alliance  points_scored    role
        5809     blue              0 defense
        9570     blue              0 defense
        3928     blue              0 defense
        8825      red              0 defense
        1736      red              0 defense
        2357      red              0 defense


,team_number,alliance,points_scored,role
0,5809,blue,0,defense
1,9570,blue,0,defense
2,3928,blue,0,defense
3,8825,red,0,defense
4,1736,red,0,defense
5,2357,red,0,defense


In [22]:
# ──────────────────────────────────────────────
# CELL 18 — Debug helpers
# ──────────────────────────────────────────────
print("Total +5 events       :", len(events))
print("Frames with robot data:", sum(1 for f in frame_data if frame_data[f]["robots"]))
print("Frames with reef data :", sum(1 for f in frame_data if frame_data[f]["reef"] is not None))
print("robot_points          :", dict(robot_points))
print("final_labels          :", final_labels)
print("track_alliance sample :", list(track_alliance.items())[:10])

Total +5 events       : 11
Frames with robot data: 1678
Frames with reef data : 1634
robot_points          : {}
final_labels          : {}
track_alliance sample : []
